In [ ]:
4

Task 4: Binary Logistic Regression & Fraud Evaluation (Credit Card Fraud Dataset)
● Exam Task:
1. Load a pre-sampled/balanced subset of credit card transaction data and
standardize the numerical predictors1819.
2. Implement the Sigmoid function and train a Logistic Regressionσ(𝑧) = 1
1+𝑒−𝑧
model.
3. Generate class predictions using a decision threshold of on the test split.0. 5
4. Compute and display the Confusion Matrix, Precision, Recall, and F1-Score,
highlighting why Recall is critical in fraud detection

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

# 1. Generate a synthetic pre-sampled/balanced credit card transaction dataset
X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=4,
    n_redundant=1,
    weights=[0.5, 0.5],
    random_state=42
)

# Split into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize numerical predictors
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

Training set shape: (800, 5)
Test set shape: (200, 5)


In [ ]:
# 2. Implement the Sigmoid function and Logistic Regression from scratch
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_logistic_regression(X, y, lr=0.1, epochs=1000):
    # Add bias column (intercept)
    intercept = np.ones((X.shape[0], 1))
    X_b = np.concatenate((intercept, X), axis=1)

    weights = np.zeros(X_b.shape[1])
    m = X.shape[0]

    for epoch in range(epochs):
        z = np.dot(X_b, weights)
        predictions = sigmoid(z)
        gradient = np.dot(X_b.T, (predictions - y)) / m
        weights -= lr * gradient

    return weights

# Train the model
weights = train_logistic_regression(X_train_scaled, y_train)
print("Model training complete. Learned weights (including intercept):", weights)

Model training complete. Learned weights (including intercept): [ 0.00487799  0.44392112 -0.24094867 -0.36316262  0.66573875  0.48024802]


In [ ]:
# 3. Generate predictions using a decision threshold of 0.5
def predict(X, weights, threshold=0.5):
    intercept = np.ones((X.shape[0], 1))
    X_b = np.concatenate((intercept, X), axis=1)
    probabilities = sigmoid(np.dot(X_b, weights))
    return (probabilities >= threshold).astype(int)

y_pred = predict(X_test_scaled, weights, threshold=0.5)

In [ ]:
# 4. Compute metrics and display performance
def compute_metrics(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    confusion_matrix = np.array([[tn, fp], [fn, tp]])
    return confusion_matrix, precision, recall, f1

cm, precision, recall, f1 = compute_metrics(y_test, y_pred)

print("Confusion Matrix:\n", cm)
print(f"Precision: {precision:.4f}")
print(f"Recall (Sensitivity): {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Confusion Matrix:
 [[71 27]
 [37 65]]
Precision: 0.7065
Recall (Sensitivity): 0.6373
F1-Score: 0.6701


### Why Recall is Critical in Fraud Detection
In fraud detection, **Recall** is highly critical because the cost of a **False Negative** (failing to detect actual fraud) is significantly higher than the cost of a **False Positive** (flagging a legitimate transaction for review). High Recall ensures that we capture as many fraudulent transactions as possible, protecting clients and financial systems.